In [1]:
%matplotlib osx
# This script is a tool to visualize laser data charge and timing distributions
# In particular this was used to compare 2023 and 2025 laser data

# In principle it can be desconstructed to analyze a single run for laser analyses

# import packages
import uproot3 as uproot
import os, sys
import numpy as np
import pandas
from tqdm import trange
import matplotlib.pyplot as plt
from tqdm import trange
import matplotlib as mpl
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

# set text style for plots
font = {'family' : 'serif', 'size' : 12 }
mpl.rc('font', **font)
mpl.rcParams['mathtext.fontset'] = 'cm' # Set the math font to Computer Modern
mpl.rcParams['legend.fontsize'] = 1


PE = []; X = []; Y = []; Z = []; T = []; Ch = []; TW = []; Q = []
file_name = []
# --------------------------------------------------- #
# Load Data (2023: 4695, 2025: 5792)

file_list = ['laser_4695_gains_integration.ntuple.root',
             'BeamCluster_5792_4_30.ntuple.root']


for file in file_list:
    
    print('Loading ' + file + '...\n')
    
    file4 = uproot.open(file)

    MC4 = file4['Event']
    X.append(MC4['hitX'].array())
    Y.append(MC4['hitY'].array())
    Z.append(MC4['hitZ'].array())
    T.append(MC4['hitT'].array())
    PE.append(MC4['hitPE'].array())
    Q.append(MC4['hitQ'].array())
    Ch.append(MC4['hitDetID'].array())
    TW.append(MC4['trigword'].array())
    file_name.append(MC4['runNumber'].array())

# # # # # # # # # # # # # # # # # # # # # # # # # # #
# LoadGeometry file

# Read Geometry.csv file to get PMT location info
df = pandas.read_csv('../FullTankPMTGeometry.csv')

channel_number = []; location = []; panel_number = []
x_position = []; y_position = []; z_position = []
PMT_type = []

# The LoadGeometry File does not share the same origin point as the WCSim output after ToolAnalysis.

# vertical center (y) is at a height of y = -14.46 cm
# x-axis is fine
# z-axis (beamline) is offset by 1.681 m
# tank center is therefore at (0,-14.46, 168.1) [cm]

for i in range(len(df['channel_num'])):   # loop over PMTs
    channel_number.append(df['channel_num'][i])
    location.append(df['detector_tank_location'][i])
    x_position.append(df['x_pos'][i]+0)
    y_position.append(df['y_pos'][i]+0.1446)
    z_position.append(df['z_pos'][i]-1.681)
    panel_number.append(df['panel_number'][i])
    PMT_type.append(df['PMT_type'][i])
        

print('done')

Loading ../laser_data/v2_Jan_2025_gains_IC_adc_fix/laser_4695_gains_integration.ntuple.root...

Loading ../../../../hold/BeamCluster_5792_4_30.ntuple.root...

done


In [2]:
# count the total triggers

abc = [0,0]; def_ = [0,0]
for i in range(len(TW)):
    for j in trange(len(TW[i])):
        if str(file_name[i][j]) == '5792':
            if TW[i][j] == 47:
                abc[0] += 1
            else:
                def_[0] += 1
        else:
            if TW[i][j] == 47:
                abc[1] += 1
            else:
                def_[1] += 1
    
print('5792')
print('We have ' + str(abc[0]/(1e6)) + ' million laser triggers (47)')
print('We have ' + str(def_[0]/(1e6)) + ' million other (!= 47)\n')
print('4695')
print('We have ' + str(abc[1]/(1e6)) + ' million laser triggers (47)')
print('We have ' + str(def_[1]/(1e6)) + ' million other (!= 47)\n')

100%|█████████████████████████████████| 15420/15420 [00:00<00:00, 441177.41it/s]

5792
We have 0.01542 million laser triggers (47)
We have 0.0 million other (!= 47)

4695
We have 4.335191 million laser triggers (47)
We have 0.0 million other (!= 47)



In [45]:
# 2023 laser details: window 450 < hitT[i][j] < 500
# 2025 laser details: window 440 < hitT[i][j] < 490
# (ensure same relative window range)


def TOF(origin, t_min, hitX, hitY, hitZ, hitT):
    c = 299792458  # [m/s]
    c = c/(4/3)    # refractive index of water

    # calculate hit timing residuals
    d_pos = np.abs(np.sqrt( (origin[0] - hitZ)**2  + \
                                (origin[1] - hitX)**2  +  (origin[2] - hitY)**2 ))
    # timing residual = t - TOF - t0
    tri = (hitT)/(1e9)  -  d_pos/c - t_min/(1e9) 
    t_res_i = tri*1e9    # in ns
    
    return t_res_i


t0 = 450           # chain times to start of laser window
origin = [0,0,0]   # laser source at center (z, x, y)
origin_updated = [0, 0, 20]   # +20cm

chan_offset = 332

times_2023 = [[] for i in range(len(channel_number))]
IDs_2023 = [[] for i in range(len(channel_number))]
charge_2023 = [[] for i in range(len(channel_number))]

times_2025 = [[] for i in range(len(channel_number))]
IDs_2025 = [[] for i in range(len(channel_number))]
charge_2025 = [[] for i in range(len(channel_number))]


# omit bad PMTs from sample (from Gian)
bpi = [2,6,11,14,15,18,21,85,100,113,114]   # by index (need to add +1 for proper PMT numbers)
bad_PMTs = []
for i in range(len(bpi)):
    bad_PMTs.append(channel_number[bpi[i]-1])

for runs in range(len(X)):
    for i in trange(len(X[runs])):
        if TW[runs][i] == 47:  # they should all be trigword 47
            for j in range(len(T[runs][i])):
                if int(file_name[runs][i]) == 5792:
                    if 440 < T[runs][i][j] < 490 and Ch[runs][i][j] not in bad_PMTs:
                        res = TOF(origin_updated, t0, X[runs][i][j], Y[runs][i][j], Z[runs][i][j], T[runs][i][j])
                        times_2025[Ch[runs][i][j] - chan_offset].append(res)
                        charge_2025[Ch[runs][i][j] - chan_offset].append(PE[runs][i][j])
                        IDs_2025[Ch[runs][i][j] - chan_offset].append(Ch[runs][i][j])
                else:
                    if 450 < T[runs][i][j] < 500 and Ch[runs][i][j] not in bad_PMTs:
                        res = TOF(origin, t0, X[runs][i][j], Y[runs][i][j], Z[runs][i][j], T[runs][i][j])
                        times_2023[Ch[runs][i][j] - chan_offset].append(res)
                        charge_2023[Ch[runs][i][j] - chan_offset].append(PE[runs][i][j])
                        IDs_2023[Ch[runs][i][j] - chan_offset].append(Ch[runs][i][j])
                    

print('done')

100%|████████████████████████████████████| 15420/15420 [01:15<00:00, 204.18it/s]

done


In [4]:
print(len(times_2025[0]))
dummy_time = [[], []]; dummy_charge = [[], []]
for i in trange(len(times_2025), desc = '2025'):          
    for j in range(len(times_2025[i])):
        if 0 < charge_2025[i][j] < 10:
            dummy_time[0].append(times_2025[i][j])
            dummy_charge[0].append(charge_2025[i][j])
for i in trange(len(times_2023), desc = '2023'):          
    for j in range(len(times_2023[i])):
        if 0 < charge_2023[i][j] < 10:
            dummy_time[1].append(times_2023[i][j])
            dummy_charge[1].append(charge_2023[i][j])

        
fig, ax = plt.subplots(1, figsize = (4,3))

plt.hist2d(dummy_time[0], dummy_charge[0], bins = (50,50))
#plt.xlim([-10,50])
plt.xlabel('nanoseconds', fontsize = 12)
plt.ylabel('photoelectrons', fontsize = 12)
plt.title('2025 data')
plt.xlim([-10, 20]); plt.ylim([0,5])
plt.savefig('plots/2025_5792/2025 5792 data 2D charge vs time.png',
            dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
plt.show()

print('done')

17399


2023: 100%|███████████████████████████████████| 132/132 [00:03<00:00, 41.15it/s]


done


In [9]:
# rate per channel
    
counts_2023 = abc[1]
counts_2025 = abc[0]

hits_per_chan_2023 = [100 * len(t) / counts_2023 for t in times_2023]
hits_per_chan_2025 = [100 * len(t) / counts_2025 for t in times_2025]
    
fig, ax = plt.subplots(1)#, figsize = (6,5))
#plt.scatter(channel_number, hits_per_chan_2023, s = 15, color = 'black', label = 'R4695 (Nov 2023)', marker = '|')
plt.scatter(channel_number, hits_per_chan_2025, s = 15, color = 'red', label = 'R5346 (Feb 2025)', marker = '_')
plt.ylabel('Hits per laser trigger in laser window')
plt.xlabel('Tube Number')
plt.yscale('log')
plt.title('old vs new laser run light metrics')
ax.legend(frameon = False, loc='lower right', fontsize = 12)
#plt.ylim([-0.01,0.10])
#ax.text(0.4,0.5,'No Hits',size=12,transform=axes[j].transAxes,fontweight = 'bold')

plt.savefig('plots/2023 vs 2025 laser data _ PMT hit yield per channel.png',
                dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
plt.show()

In [13]:
# total occupancy (total PMT hits per trigger)
    
# total triggers
counts_2023 = abc[1]
counts_2025 = abc[0]

hits_per_chan_2023 = [len(t) for t in times_2023]
hits_per_chan_2025 = [len(t) for t in times_2025]

hits_per_chan_2023 = 100 * sum(hits_per_chan_2023) / counts_2023 / 121
hits_per_chan_2025 = 100 * sum(hits_per_chan_2025) / counts_2025 / 121

years = ['2023', '2025']
values = [hits_per_chan_2023, hits_per_chan_2025]
colors = ['black', 'red']
    
#plt.scatter(5, hits_per_chan_2023, s = 15, color = 'black', label = '2023: ' + str(round(hits_per_chan_2023,2)) + ' %')
#plt.scatter(5, hits_per_chan_2025, s = 15, color = 'red', label = '2025: ' + str(round(hits_per_chan_2025,2)) + ' %')
fig, ax = plt.subplots(1, figsize = (4,3))
ax.barh(years, values, color=colors, height=0.4)

plt.xlabel('% of total PMTs hit per laser trigger')
plt.title('Laser PMT Occupancy')

for i, v in enumerate(values):
    ax.text(v + 2, i, f"{v:.2f}%", va='center', fontsize=12)

plt.xlim(0, 150)
plt.savefig('plots/2025_5792/2023 vs 2025 5792 laser data _ PMT occupancy.png',
                dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
plt.show()

In [47]:
# Look at timing distribution (After TOF corrections)

dummy_time = [[], []]; dummy_charge = [[], []]
for i in trange(len(times_2025), desc = '2025'):          
    for j in range(len(times_2025[i])):
        if 0 < charge_2025[i][j] < 20 and times_2025[i][j] < 25:
            dummy_time[0].append(times_2025[i][j])
            dummy_charge[0].append(charge_2025[i][j])
for i in trange(len(times_2023), desc = '2023'):          
    for j in range(len(times_2023[i])):
        if 0 < charge_2023[i][j] < 10 and times_2023[i][j] < 35:
            dummy_time[1].append(times_2023[i][j])
            dummy_charge[1].append(charge_2023[i][j])

            
fig, ax = plt.subplots(1, figsize = (4,3))

mini1 = min(dummy_time[1]); maxi1 = max(dummy_time[1]); range1 = np.abs(maxi1) + np.abs(mini1)
mini2 = min(dummy_time[0]); maxi2 = max(dummy_time[0]); range2 = np.abs(maxi2) + np.abs(mini2)
ratio = range1/range2
print('\n2023:', range1, '\n2025:', range2, '\nratio:', ratio)

plt.hist(dummy_time[1], bins=int(100*ratio), color='navy', label = '2023', linewidth=1.5, histtype='step', density = True)
plt.hist(dummy_time[0], bins=100, color='red', label = '2025', linewidth=1.5, histtype='step', density = True)
#plt.xlim([-10,50])
plt.xlabel('nanoseconds', loc = 'right', fontsize = 12)
plt.title('PMT laser timing distribution')
plt.legend(fontsize=12, frameon=False)
plt.savefig('plots/2025_5792/laser timing distribution _ after TOF corrections _ 2023 (4695) vs 2025 (5792).png',
            dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
plt.show()

print('done')

2023: 100%|███████████████████████████████████| 132/132 [00:03<00:00, 35.32it/s]



2023: 42.69432261917698 
2025: 147.74209133586814 
ratio: 0.2889787347203461
done


In [19]:
fig, ax = plt.subplots(1, figsize = (4,3))

mini1 = min(dummy_charge[1]); maxi1 = max(dummy_charge[1]); range1 = np.abs(maxi1) + np.abs(mini1)
mini2 = min(dummy_charge[0]); maxi2 = max(dummy_charge[0]); range2 = np.abs(maxi2) + np.abs(mini2)
ratio = range1/range2
print('\n2023:', range1, '\n2025:', range2, '\nratio:', ratio)

plt.hist(dummy_charge[1], bins=int(100*ratio), color='navy', label = '2023', linewidth=1.5, histtype='step', density = True)
plt.hist(dummy_charge[0], bins=100, color='red', label = '2025', linewidth=1.5, histtype='step', density = True)
#plt.xlim([-10,50])
plt.xlabel('photoelectrons', loc = 'right', fontsize = 12)
plt.title('PMT laser charge per hit')
plt.legend(fontsize=12, frameon=False)
plt.savefig('plots/2025_5792/laser charge distribution _ all times within 35ns laser window _ 2023 (4695) vs 2025 (5792).png',
            dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
plt.show()

print('done')


2023: 9.9989112647804 
2025: 10.000627986836221 
ratio: 0.9998283385745295
done
